## Imports & setting up:

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.colors import LinearSegmentedColormap

from matplotlib.lines import Line2D
import scipy.stats as scipy_stats
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# --- plot KDEs ---
COL_LABELS = {
    "t2m": "2m temperature",
    "tp": "Precipitation",
    "slhf": "Latent heat flux",
    "sshf": "Sensible heat flux",
    "ssrd": "Solar radiation",
    "fal": "Albedo",
    "str": "Thermal radiation",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "slope": "Slope (°)",
    "svf": "Sky-view factor",
}

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")


In [ ]:
# -----------------------------------------------------------------------
# Nature-compliant colorblind-safe palette
# Based on: https://research-figure-guide.nature.com/figures/building-and-exporting-figure-panels/
# -----------------------------------------------------------------------

NATURE_PALETTE = {
    "black": "#000000",
    "orange": "#e69f00",
    "sky_blue": "#56b4e9",
    "bluish_green": "#009e73",
    "yellow": "#f0e442",
    "blue": "#0072b2",
    "vermillion": "#d55e00",
    "reddish_purple": "#cc79a7",
}

# Convenient ordered list for sequential assignment
NATURE_COLORS = list(NATURE_PALETTE.values())

# Nature figure specs (for reference when setting figsize)
NATURE_SPECS = {
    "single_col_mm": 89,
    "double_col_mm": 183,
    "max_height_mm": 170,
    "font_min_pt": 5,
    "font_max_pt": 7,
    "font_family": "Arial",
    "dpi": 300,
}


def nature_figsize(cols=1, height_mm=80):
    """Return figsize in inches for Nature single or double column."""
    width_mm = NATURE_SPECS["single_col_mm"] if cols == 1 \
               else NATURE_SPECS["double_col_mm"]
    return (width_mm / 25.4, height_mm / 25.4)


def apply_nature_style(ax, fontsize=6, box=False):
    if box:
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.4)
    else:
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["bottom", "left"]].set_linewidth(0.5)
    ax.tick_params(labelsize=fontsize, width=0.5, length=3)
    ax.xaxis.label.set_size(fontsize)
    ax.yaxis.label.set_size(fontsize)
    ax.title.set_size(fontsize + 1)
    ax.grid(color="#e0e0e0", linewidth=0.3, zorder=0)
    ax.set_axisbelow(True)


mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 6,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.width": 0.5,
    "ytick.major.size": 3,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

nature_seq = LinearSegmentedColormap.from_list(
    "nature_seq",
    [
        NATURE_PALETTE["sky_blue"], NATURE_PALETTE["blue"],
        NATURE_PALETTE["black"]
    ],
)


## Load glacier grids:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head(2)

### Switzerland:

In [ ]:
df_CH = load_stakes(cfg, "CH")
print(df_CH.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'aletsch'
rgi_id_aletsch = df_CH[df_CH.GLACIER == glacier_name].RGIId.values[0]

path_monthly_dfs = os.path.join(
    cfg.dataPath, 'RGI_v6/RGI_11_CentralEurope/monthly_parquets/')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_aletsch)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_aletsch}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_aletsch = pd.concat([pd.read_parquet(f) for f in yearly_files],
                       ignore_index=True)
df_aletsch.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_aletsch.shape}")

# df = df_aletsch[df_aletsch.YEAR == 2016]
# fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=70))

# # Map the 'aspect' categories to Nature colors
# aspects = df["aspect"].dropna().unique()

# sns.scatterplot(
#     data=df,
#     x="POINT_LON",
#     y="POINT_LAT",
#     hue="aspect",
#     palette=nature_seq,
#     s=4,  # small markers suit Nature's dense figures
#     linewidth=0,  # no edge — cleaner at small sizes
#     alpha=0.8,
#     ax=ax,
# )

# apply_nature_style(ax, fontsize=6)  # strip top/right spines, grid, etc.

# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# ax.set_title(f"Aletsch – aspect (2016)", pad=4)

# # Tighten legend
# leg = ax.legend(
#     title="Aspect",
#     title_fontsize=6,
#     fontsize=5,
#     frameon=False,
#     markerscale=1.5,
#     handletextpad=0.4,
#     labelspacing=0.3,
# )

# plt.tight_layout()
# plt.show()

### Alaska:

In [ ]:
df_ALA = load_stakes(cfg, "ALA")
print(df_ALA.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'KAHILTNA'
rgi_id_kahil = df_ALA[df_ALA.GLACIER == glacier_name].RGIId.values[0]
print(rgi_id_kahil)
# rgi_id_kahil = "RGI60-01.00013"
# glacier_name = df_ALA[df_ALA.RGIId == rgi_id_kahil].GLACIER.values[0]
print(
    f"Loading monthly data for glacier {glacier_name} with RGI ID {rgi_id_kahil}"
)

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_01_Alaska/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_kahil)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_kahil}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_gl_alaska = pd.concat([pd.read_parquet(f) for f in yearly_files],
                         ignore_index=True)
df_gl_alaska.dropna(subset=feature_columns, inplace=True)
print(f"Shape: {df_gl_alaska.shape}")

df = df_gl_alaska[df_gl_alaska.YEAR == 2016]
fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=70))

# Map the 'aspect' categories to Nature colors
aspects = df["aspect"].dropna().unique()

sns.scatterplot(
    data=df,
    x="POINT_LON",
    y="POINT_LAT",
    hue="aspect",
    palette=nature_seq,
    s=4,  # small markers suit Nature's dense figures
    linewidth=0,  # no edge — cleaner at small sizes
    alpha=0.8,
    ax=ax,
)

apply_nature_style(ax, fontsize=6)  # strip top/right spines, grid, etc.

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"KAHILTNA – aspect (2016)", pad=4)

# Tighten legend
leg = ax.legend(
    title="Aspect",
    title_fontsize=6,
    fontsize=5,
    frameon=False,
    markerscale=1.5,
    handletextpad=0.4,
    labelspacing=0.3,
)

plt.tight_layout()
plt.show()

### Norway:

In [ ]:
df_NOR = load_stakes(cfg, "NOR")
print(df_NOR.GLACIER.unique())

# --- load aletsch monthly grid ---
glacier_name = 'Aalfotbreen'
rgi_id_aalfot = df_NOR[df_NOR.GLACIER == glacier_name].RGIId.values[0]

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_08_Scandinavia/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_aalfot)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_aalfot}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_gl_nor = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
df_gl_nor.dropna(subset=feature_columns, inplace=True)
# print(f"Shape: {df_gl_nor.shape}")

# df = df_gl_nor[df_gl_nor.YEAR == 2016]
# fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=70))

# # Map the 'aspect' categories to Nature colors
# aspects = df["aspect"].dropna().unique()

# sns.scatterplot(
#     data=df,
#     x="POINT_LON",
#     y="POINT_LAT",
#     hue="aspect",
#     palette=nature_seq,
#     s=4,  # small markers suit Nature's dense figures
#     linewidth=0,  # no edge — cleaner at small sizes
#     alpha=0.8,
#     ax=ax,
# )

# apply_nature_style(ax, fontsize=6)  # strip top/right spines, grid, etc.

# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# ax.set_title(f"{glacier_name} – aspect (2016)", pad=4)

# # Tighten legend
# leg = ax.legend(
#     title="Aspect",
#     title_fontsize=6,
#     fontsize=5,
#     frameon=False,
#     markerscale=1.5,
#     handletextpad=0.4,
#     labelspacing=0.3,
# )

# plt.tight_layout()
# plt.show()

### Iceland:

In [ ]:
df_ISL = load_stakes(cfg, "ISL")
print(df_ISL.GLACIER.unique())

# --- load aletsch monthly grid ---
rgi_id_ISL = "RGI60-06.00340"
glacier_name = df_ISL[df_ISL.RGIId == rgi_id_ISL].GLACIER.values[0]

path_monthly_dfs = os.path.join(cfg.dataPath,
                                'RGI_v6/RGI_06_Iceland/monthly_parquets')
path_parquet = os.path.join(path_monthly_dfs, rgi_id_ISL)

yearly_files = sorted(
    glob.glob(os.path.join(path_parquet, f"{rgi_id_ISL}_grid_*.parquet")))
print(f"Found {len(yearly_files)} yearly files for {glacier_name}")

df_gl_ISL = pd.concat([pd.read_parquet(f) for f in yearly_files],
                      ignore_index=True)
df_gl_ISL.dropna(subset=feature_columns, inplace=True)

### DEMS:

In [ ]:
# ── 0. Global elevation range across all three glaciers ──────────────────
datasets = {
    "aletsch":
    cfg.dataPath +
    f"/RGI_v6/RGI_11_CentralEurope/xr_masked_grids/{rgi_id_aletsch}.zarr",
    "kahiltna":
    cfg.dataPath +
    f"/RGI_v6/RGI_01_Alaska/xr_masked_grids/{rgi_id_kahil}.zarr",
    "aalfot":
    cfg.dataPath +
    f"/RGI_v6/RGI_08_Scandinavia/xr_masked_grids/{rgi_id_aalfot}.zarr",
}

vmin, vmax = np.inf, -np.inf
for path in datasets.values():
    elev = xr.open_dataset(path).masked_elev
    # match robust=True: use 2nd–98th percentile
    vmin = min(vmin, float(elev.quantile(0.02)))
    vmax = max(vmax, float(elev.quantile(0.98)))

print(f"Global elevation range: {vmin:.0f} – {vmax:.0f} m")

elev_cmap = plt.cm.terrain
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

In [ ]:
# ── 1. Reusable plot function ─────────────────────────────────────────────
def plot_dem(path, title, fname):
    ds = xr.open_dataset(path)
    fig, ax = plt.subplots(figsize=nature_figsize(cols=1, height_mm=80))

    ds.masked_elev.plot(
        ax=ax,
        cmap=elev_cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False,
    )

    apply_nature_style(ax, fontsize=6)

    # override apply_nature_style — restore full box
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(length=0)
    ax.set_title(title, pad=4, fontsize=7)

    plt.tight_layout()
    plt.savefig(f"figures/paperTF/{fname}.pdf", bbox_inches="tight")
    # plt.savefig(f"figures/paperTF/{fname}.png", dpi=300, bbox_inches="tight")
    plt.show()


plot_dem(datasets["aletsch"], "", "aletsch_DEM")
plot_dem(datasets["kahiltna"], "", "kahiltna_DEM")
plot_dem(datasets["aalfot"], "", "aalfot_DEM")

In [ ]:
# ── 3-panel DEM figure (same height as KDE plot) ─────────────────────────
dem_order = [
    ("aalfot", "Ålfotbreen (NOR)"),
    ("aletsch", "Grosser Aletschgletscher (CH)"),
    ("kahiltna", "Kahiltna Glacier (US)"),
]

fig_dem, axes_dem = plt.subplots(
    3,
    1,
    figsize=(60 / 25.4, 200 / 25.4),  # narrow, same height as KDE
)

for ax, (key, title) in zip(axes_dem, dem_order):
    ds = xr.open_dataset(datasets[key])
    ds.masked_elev.plot(
        ax=ax,
        cmap=elev_cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False,
    )

    apply_nature_style(ax, fontsize=6, box=True)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(length=0)
    ax.set_title(title, pad=3, fontsize=6, loc="left")

plt.tight_layout(pad=0.5, h_pad=1.0)
plt.savefig("figures/paperTF/all_DEMs.pdf", bbox_inches="tight")
# plt.savefig("figures/paperTF/all_DEMs.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.colorbar as mcolorbar
# ── 2. Standalone colorbar ────────────────────────────────────────────────
fig_cb, ax_cb = plt.subplots(figsize=(2, 0.2))  # wide & short

cb = mcolorbar.ColorbarBase(
    ax_cb,
    cmap=elev_cmap,
    norm=norm,
    orientation="horizontal",
)
cb.set_label("Elevation (m a.s.l.)", fontsize=6)
cb.ax.tick_params(labelsize=5, width=0.5, length=3)
cb.outline.set_linewidth(0.5)

plt.tight_layout()
plt.savefig("figures/paperTF/shared_colorbar.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/shared_colorbar.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

In [ ]:
# NOR_GLACIERS = {
#     "Ålfotbreen": "Aalfotbreen",
#     "Hansebreen": "Hansebreen",
#     "Nigardsbreen": "Nigardsbreen",
#     "Engabreen": "Engabreen",
# }

# df_nor_by_glacier = {}
# for display_name, glacier_name in NOR_GLACIERS.items():
#     try:
#         rgi_id_gl = df_NOR[df_NOR.GLACIER == glacier_name].RGIId.values[0]
#     except IndexError:
#         print(f"Not found: {glacier_name}")
#         continue

#     path_parquet = os.path.join(cfg.dataPath,
#                                 'RGI_v6/RGI_08_Scandinavia/monthly_parquets',
#                                 rgi_id_gl)
#     yearly_files = sorted(
#         glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))

#     if not yearly_files:
#         print(f"No files found for {glacier_name}")
#         continue

#     df_gl = pd.concat([pd.read_parquet(f) for f in yearly_files],
#                       ignore_index=True)
#     df_gl.dropna(subset=feature_columns, inplace=True)
#     df_nor_by_glacier[display_name] = df_gl
#     print(f"Loaded {display_name}: {df_gl.shape}")

# ALA_GLACIERS = {
#     "Gulkana": "GULKANA",
#     "Lemon Creek": "LEMON CREEK",
#     "Taku": "TAKU",
#     "Wolverine": "WOLVERINE",
#     "Kahiltna": "KAHILTNA",
# }

# df_ala_by_glacier = {}
# for display_name, glacier_name in ALA_GLACIERS.items():
#     try:
#         rgi_id_gl = df_ALA[df_ALA.GLACIER == glacier_name].RGIId.values[0]
#     except IndexError:
#         print(f"Not found: {glacier_name}")
#         continue

#     path_parquet = os.path.join(cfg.dataPath,
#                                 'RGI_v6/RGI_01_Alaska/monthly_parquets',
#                                 rgi_id_gl)
#     yearly_files = sorted(
#         glob.glob(os.path.join(path_parquet, f"{rgi_id_gl}_grid_*.parquet")))

#     if not yearly_files:
#         print(f"No files found for {glacier_name}")
#         continue

#     df_gl = pd.concat([pd.read_parquet(f) for f in yearly_files],
#                       ignore_index=True)
#     df_gl.dropna(subset=feature_columns, inplace=True)
#     df_ala_by_glacier[display_name] = df_gl
#     print(f"Loaded {display_name}: {df_gl.shape}")

### KDE plot:

In [ ]:
GLACIERS_TO_PLOT = {
    "Aletsch (CH)": {
        "df":
        df_aletsch,
        "color":
        NATURE_PALETTE["blue"],
        "rgi":
        "RGI60-11.01450",
        "extent": (5.8, 11, 45.5, 48.0),
        "gdf":
        gpd.read_file(
            os.path.join(
                cfg.dataPath,
                "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp")),
    },
    "Ålfotbreen (NOR)": {
        "df":
        df_gl_nor,
        "color":
        NATURE_PALETTE["vermillion"],
        "rgi":
        "RGI60-08.02992",
        "extent": (4.0, 24.0, 58.0, 71.0),
        "gdf":
        gpd.read_file(
            os.path.join(
                cfg.dataPath,
                "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp")),
    },
    "Kahiltna (ALA)": {
        "df":
        df_gl_alaska,
        "color":
        NATURE_PALETTE["bluish_green"],
        "rgi":
        "RGI60-01.00013",
        "extent": (-170, -142, 55.0, 68.0),
        "gdf":
        gpd.read_file(
            os.path.join(cfg.dataPath,
                         "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp")),
    },
}

# --- convenience alias for KDE plots ---
COLORS = {label: cfg_gl["color"] for label, cfg_gl in GLACIERS_TO_PLOT.items()}

In [ ]:
# selected features for the KDE panel
SELECTED_COLS = [
    "t2m", "tp", "ssrd", "sshf", "ELEVATION_DIFFERENCE", "aspect", "slope",
    "svf"
]
COL_LABELS = {
    "t2m": "2m temperature (°C)",
    "tp": "Precipitation (m/month)",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "ssrd": "Solar radiation (W/m²)",
    "svf": "Sky-view factor",
    "slope": "Slope (°)",
    "sshf": "Sensible heat flux (W/m²)",
}

import matplotlib as mpl
from matplotlib.ticker import FuncFormatter

# font fallback — use DejaVu Sans if Arial not available
try:
    mpl.font_manager.findfont("Arial", fallback_to_default=False)
    mpl.rcParams["font.family"] = "Arial"
except:
    mpl.rcParams["font.family"] = "DejaVu Sans"  # closest available


def format_axis_ticks(ax):
    """Format tick labels to avoid huge 1e6/1e7 offset labels."""
    # check if scientific notation offset is being used
    ax.xaxis.get_major_formatter().set_useOffset(False)
    try:
        # scale large numbers to readable units
        xmax = abs(ax.get_xlim()[1])
        if xmax > 1e6:
            scale = 1e6
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.1f}"))
            ax.set_xlabel(f"(×10⁶)", fontsize=5, labelpad=1)
        elif xmax > 1e4:
            scale = 1e3
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.0f}"))
            ax.set_xlabel(f"(×10³)", fontsize=5, labelpad=1)
    except Exception:
        pass


ncols = 2
nrows = int(np.ceil(len(SELECTED_COLS) / ncols))

# fig, axes = plt.subplots(
#     nrows,
#     ncols,
#     figsize=nature_figsize(cols=2, height_mm=90),
#     squeeze=False,
# )

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(130 / 25.4, 200 / 25.4),  # single column width, tall
    squeeze=False,
)

# COLORS already derived from GLACIERS_TO_PLOT above

for idx, col in enumerate(SELECTED_COLS):
    ax = axes[idx // ncols][idx % ncols]

    all_vals = pd.concat(
        [cfg_gl["df"][col].dropna() for cfg_gl in GLACIERS_TO_PLOT.values()])
    xmin = float(all_vals.min())
    xmax = float(all_vals.max())
    x_grid = np.linspace(xmin, xmax, 500)

    for label, cfg_gl in GLACIERS_TO_PLOT.items():
        vals = cfg_gl["df"][col].dropna().values
        if len(vals) < 10:
            continue
        kde = scipy_stats.gaussian_kde(vals)
        y = kde(x_grid)
        y = y / y.max()
        ax.plot(x_grid, y, color=cfg_gl["color"], linewidth=0.8, label=label)
        ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

    ax.set_ylim(0, 1.15)
    ax.set_title(COL_LABELS.get(col, col), fontsize=6, pad=3)
    ax.set_xlabel("")
    #ax.set_yticks([])
    ax.tick_params(labelsize=5, width=0.4, length=2)
    # ax.spines[["top", "right", "left"]].set_visible(False)
    # ax.spines["bottom"].set_linewidth(0.4)
    ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
    ax.set_axisbelow(True)

    format_axis_ticks(ax)

for idx in range(len(SELECTED_COLS), nrows * ncols):
    axes[idx // ncols][idx % ncols].axis("off")

handles = [
    plt.Line2D([0], [0], color=COLORS[l], linewidth=1.0, label=l)
    for l in GLACIERS_TO_PLOT
]
fig.legend(
    handles=handles,
    loc="lower center",
    ncol=len(GLACIERS_TO_PLOT),
    frameon=False,
    fontsize=6,
    bbox_to_anchor=(0.5, 0.03),  # ← raise slightly so it doesn't clip
)

plt.tight_layout(
    pad=0.8,
    h_pad=3.0,
    w_pad=0.8,
    rect=[0, 0.05, 1, 0.97],  # ← was 0.06, reduce to shrink the gap
)
plt.savefig("figures/paperTF/section1_kde_nature.pdf", bbox_inches="tight")
plt.savefig("figures/paperTF/section1_kde_nature.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

### Maps:

In [ ]:
def plot_glacier_outline_map(
    glacier_outline_rgi,
    highlight_rgi_id=None,
    highlight_color="#2166ac",
    *,
    title="",
    figsize=nature_figsize(cols=1, height_mm=55),
    extent=(5.8, 15, 44, 48),
    land_facecolor="#f0f0f0",
    land_alpha=1.0,
    outline_color="#555555",
    outline_alpha=0.6,
    add_features=True,
    add_gridlines=False,
    show=True,
):
    lonW, lonE, latS, latN = extent
    projPC = ccrs.PlateCarree()

    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection=projPC)
    ax.set_extent([lonW, lonE, latS, latN], crs=ccrs.Geodetic())

    if add_features:
        ax.add_feature(cfeature.OCEAN, facecolor="#daeeff", zorder=0)
        ax.add_feature(cfeature.LAND,
                       facecolor=land_facecolor,
                       alpha=land_alpha,
                       zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.3, zorder=1)
        ax.add_feature(cfeature.BORDERS,
                       linestyle="-",
                       linewidth=0.2,
                       zorder=1)
        ax.add_feature(cfeature.LAKES, linewidth=0.2, zorder=1)
        ax.add_feature(cfeature.RIVERS, linewidth=0.2, zorder=1)

    if add_gridlines:
        gl = ax.gridlines(draw_labels=True,
                          linewidth=0.2,
                          color="gray",
                          alpha=0.4,
                          linestyle="--")
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = {"size": 5}
        gl.ylabel_style = {"size": 5}

    # --- all glacier outlines clipped to extent ---
    gdf_clipped = glacier_outline_rgi.cx[lonW:lonE, latS:latN]
    if not gdf_clipped.empty:
        gdf_clipped.plot(ax=ax,
                         transform=projPC,
                         color=outline_color,
                         alpha=outline_alpha,
                         linewidth=0.2)

    # --- star marker on target glacier ---
    if highlight_rgi_id is not None:
        gdf_target = glacier_outline_rgi[glacier_outline_rgi["RGIId"] ==
                                         highlight_rgi_id]
        if not gdf_target.empty:
            centroid = gdf_target.geometry.centroid.iloc[0]
            ax.plot(
                centroid.x,
                centroid.y,
                marker="*",
                markersize=20,
                color=highlight_color,
                markeredgecolor="white",
                markeredgewidth=0.3,
                transform=projPC,
                zorder=4,
            )
        else:
            print(f"Warning: RGIId {highlight_rgi_id} not found in shapefile.")

    if title:
        ax.set_title(title,
                     fontsize=20,
                     fontweight="bold",
                     loc="left",
                     pad=2,
                     color=highlight_color)

    plt.tight_layout(pad=0.2)
    if show:
        plt.show()

    return fig, ax

In [ ]:
for label, cfg_gl in GLACIERS_TO_PLOT.items():
    fig, ax = plot_glacier_outline_map(
        glacier_outline_rgi=cfg_gl["gdf"],
        highlight_rgi_id=cfg_gl["rgi"],
        highlight_color=cfg_gl["color"],
        title=label,
        extent=cfg_gl["extent"],
        figsize=(6, 5),
        show=False,
    )
    fname = label.replace(" ", "_").replace("(",
                                            "").replace(")",
                                                        "").replace("/", "_")
    plt.savefig(f"figures/paperTF/map_{fname}.pdf", bbox_inches="tight")
    plt.savefig(f"figures/paperTF/map_{fname}.png",
                dpi=300,
                bbox_inches="tight")
    plt.close()
    print(f"Saved map_{fname}")